<a href="https://colab.research.google.com/github/vectara/example-notebooks/blob/main/notebooks/api-examples/11-agent-schedules.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Agent Schedules: Automating Agent Execution

This notebook demonstrates how to use Vectara's **Agent Schedules** to automate agent execution on recurring intervals. Schedules enable agents to run automatically without user interaction, making them ideal for:

- **Periodic reports**: Generate daily/weekly summaries from your corpus
- **Monitoring**: Regularly check for new information or changes
- **Data processing**: Automated extraction or analysis on a schedule
- **Alerting**: Periodic scans that could trigger downstream actions

You'll learn how to:
1. Create schedules with cron expressions and interval-based timing
2. Monitor schedule execution history
3. Update and manage schedule lifecycle
4. Inspect sessions created by scheduled executions

## About Vectara

[Vectara](https://vectara.com/) is an Agent Platform for trusted enterprise AI — a unified Agentic RAG platform with built-in retrieval, orchestration, and governance. See [Notebook 1](1-corpus-creation.ipynb) for the full overview of features and deployment options.

## Getting Started

This notebook assumes you've completed Notebooks 1 and 2:
- Notebook 1: Created two corpora (ai-research-papers and vectara-docs) with Boomerang embeddings
- Notebook 2: Ingested AI research papers and Vectara documentation

## Setup

In [1]:
import os
import requests
import json

# Set up authentication
api_key = os.environ['VECTARA_API_KEY']

# Corpus keys from Notebook 1
research_corpus_key = 'tutorial-ai-research-papers'

# Base URL for Vectara API v2
BASE_URL = "https://api.vectara.io/v2"

# Common headers (matches every other api-example notebook).
headers = {
    "x-api-key": api_key,
    "Content-Type": "application/json",
}

print("Setup complete.")

Setup complete.


In [2]:
# Load the shared helpers (delete_and_create_agent).
# vectara_utils.py sits next to this notebook in the repo; Colab fetches it on
# first run so the same notebook works locally and in a fresh Colab runtime.
try:
    from vectara_utils import delete_and_create_agent
except ImportError:
    import urllib.request
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/vectara/example-notebooks/main/notebooks/api-examples/vectara_utils.py",
        "vectara_utils.py",
    )
    from vectara_utils import delete_and_create_agent

import vectara_utils
vectara_utils.configure(BASE_URL, headers)


## Step 1: Create an Agent for Scheduled Execution

First, create an agent designed for automated execution. This agent generates a research digest from the corpus.

We use the modern `steps` + `first_step_name` pattern (not the deprecated `first_step`).

In [3]:
agent_name = "Research Digest Generator"
agent_config = {
    "name": agent_name,
    "description": "Agent that generates periodic research digests from indexed papers",
    "model": {"name": "gpt-4o"},
    "first_step_name": "generate_digest",
    "steps": {
        "generate_digest": {
            "instructions": [
                {
                    "type": "inline",
                    "name": "digest_instructions",
                    "template": """You are a research digest generator. When triggered, search the research corpus
for key findings and produce a concise digest covering the main themes.

Format your response as a structured digest with:
- A brief executive summary (2-3 sentences)
- Top 3-5 key findings or themes
- Any notable trends or connections between papers

Always ground your response in retrieved content with citations."""
                }
            ],
            "output_parser": {"type": "default"}
        }
    },
    "tool_configurations": {
        "research_search": {
            "type": "corpora_search",
            "query_configuration": {
                "search": {
                    "corpora": [{"corpus_key": research_corpus_key}],
                    "limit": 20
                }
            }
        }
    }
}

agent_key = delete_and_create_agent(agent_config, agent_name)


Created agent 'Research Digest Generator' (key: agt_research_digest_generator_81da)


## Step 2: Create a Cron-Based Schedule

Cron schedules use standard 5-field cron expressions to define when the agent should run.

| Field | Values | Example |
|-------|--------|--------|
| Minute | 0-59 | `0` = top of hour |
| Hour | 0-23 | `9` = 9 AM |
| Day of month | 1-31 | `*` = every day |
| Month | 1-12 | `*` = every month |
| Day of week | 0-6 (Sun=0) | `1-5` = Mon-Fri |

Use [crontab.guru](https://crontab.guru/) to test and validate cron expressions.

In [4]:
if not agent_key:
    print("Agent not available. Run Step 1 first.")
else:
    # Create a daily schedule that runs every weekday morning at 9 AM UTC
    cron_schedule_config = {
        "name": "Daily Research Digest",
        "description": "Generates a research digest every weekday morning",
        "message": [
            {
                "type": "text",
                "content": "Generate today's research digest. Focus on the latest findings "
                           "about RAG, embeddings, and retrieval techniques."
            }
        ],
        "schedule": {
            "type": "cron",
            "cron_expression": "0 9 * * 1-5"  # Weekdays at 9:00 AM UTC
        },
        "enabled": False,  # Start disabled for demo purposes
        "session_metadata": {
            "report_type": "daily_digest",
            "automated": "true"
        },
        "max_executions_to_keep": 20
    }

    url = f"{BASE_URL}/agents/{agent_key}/schedules"
    response = requests.post(url, headers=headers, json=cron_schedule_config)

    cron_schedule_key = None
    if response.status_code == 201:
        schedule_data = response.json()
        cron_schedule_key = schedule_data["key"]
        print(f"Schedule created: {schedule_data['name']}")
        print(f"  Key: {cron_schedule_key}")
        print(f"  Cron: {schedule_data['schedule']['cron_expression']}")
        print(f"  Enabled: {schedule_data['enabled']}")
        print(f"  Max executions to keep: {schedule_data.get('max_executions_to_keep', 'N/A')}")
    else:
        print(f"Error: {response.text}")

Schedule created: Daily Research Digest
  Key: asc_daily_research_digest_a99e
  Cron: 0 9 * * 1-5
  Enabled: False
  Max executions to keep: 20


## Step 3: Create an Interval-Based Schedule

Interval schedules use ISO-8601 duration format (e.g., `PT6H` for 6 hours, `PT24H` for 24 hours).
The minimum interval is `PT1H` (1 hour).

In [5]:
interval_schedule_key = None
if not agent_key:
    print("Agent not available. Run Step 1 first.")
else:
    # Create an interval-based schedule that runs every 6 hours
    interval_schedule_config = {
        "name": "Periodic Research Check",
        "description": "Checks for research updates every 6 hours",
        "message": [
            {
                "type": "text",
                "content": "Briefly summarize the top 3 most relevant findings about "
                           "transformer architectures from the research corpus."
            }
        ],
        "schedule": {
            "type": "interval",
            "interval": "PT6H"  # Every 6 hours (ISO-8601 duration)
        },
        "enabled": False,  # Start disabled for demo purposes
        "session_metadata": {
            "report_type": "periodic_check"
        },
        "max_executions_to_keep": 10
    }

    url = f"{BASE_URL}/agents/{agent_key}/schedules"
    response = requests.post(url, headers=headers, json=interval_schedule_config)

    if response.status_code == 201:
        schedule_data = response.json()
        interval_schedule_key = schedule_data["key"]
        print(f"Schedule created: {schedule_data['name']}")
        print(f"  Key: {interval_schedule_key}")
        print(f"  Interval: {schedule_data['schedule']['interval']}")
        print(f"  Enabled: {schedule_data['enabled']}")
    else:
        print(f"Error: {response.text}")

Schedule created: Periodic Research Check
  Key: asc_periodic_research_check_c53e
  Interval: PT6H
  Enabled: False


## Step 4: List and Inspect Schedules

Retrieve all schedules for an agent to see their configuration and status.

In [6]:
import time

# Schedule creation is eventually consistent, so poll the list endpoint
# until the keys we just created show up (or we hit the timeout).
url = f"{BASE_URL}/agents/{agent_key}/schedules"
expected_keys = {k for k in (cron_schedule_key, interval_schedule_key) if k}

schedules = []
body = {}
deadline = time.time() + 30  # up to 30s
while time.time() < deadline:
    response = requests.get(url, headers=headers)
    if response.status_code != 200:
        print(f"Error: {response.status_code} - {response.text}")
        break
    body = response.json()
    schedules = body.get('schedules', [])
    found_keys = {s['key'] for s in schedules}
    if expected_keys.issubset(found_keys):
        break
    time.sleep(2)

if response.status_code == 200:
    print(f"Found {len(schedules)} schedule(s):\n")
    for schedule in schedules:
        stype = schedule['schedule'].get('type', 'unknown')
        timing = (schedule['schedule'].get('cron_expression') or
                  schedule['schedule'].get('interval', 'N/A'))
        print(f"  {schedule['name']}")
        print(f"    Key: {schedule['key']}")
        print(f"    Type: {stype} ({timing})")
        print(f"    Enabled: {schedule['enabled']}")
        print(f"    Last execution: {schedule.get('last_execution_at', 'Never')}")
        print()

    # If list still came back short of what we created, surface the raw body
    # so it's clear what the server actually returned.
    missing = expected_keys - {s['key'] for s in schedules}
    if missing:
        print(f"Expected schedules {sorted(missing)} did not appear within the timeout.")
        print("Raw response body:")
        print(json.dumps(body, indent=2))

Found 2 schedule(s):

  asc_periodic_research_check_c53e
    Key: asc_periodic_research_check_c53e
    Type: interval (PT6H)
    Enabled: False
    Last execution: Never

  asc_daily_research_digest_a99e
    Key: asc_daily_research_digest_a99e
    Type: interval (PT1H)
    Enabled: False
    Last execution: Never



## Step 5: Update a Schedule

Schedules can be updated with `PATCH`. Here we enable the cron schedule and switch its timing from weekdays-at-9-AM to hourly, so an execution actually fires during this tutorial. We intentionally leave the display name (`Daily Research Digest`) unchanged — a `PATCH` only needs to carry the fields you're changing, and cadence changes don't require a rename.

In [7]:
# Enable the cron schedule and update its timing to run every hour
update_config = {
    "enabled": True,
    "schedule": {
        "type": "cron",
        "cron_expression": "0 * * * *"  # Every hour (for demo purposes)
    },
    "description": "Generates a research digest every hour (demo)"
}

url = f"{BASE_URL}/agents/{agent_key}/schedules/{cron_schedule_key}"
response = requests.patch(url, headers=headers, json=update_config)

if response.status_code == 200:
    updated = response.json()
    # Some list/patch response shapes report the resource identifier under
    # `name`; prefer a human label if the server provides one.
    label = updated.get('display_name') or updated.get('name') or cron_schedule_key
    print(f"Schedule updated: {label}")
    print(f"  Key: {updated.get('key', cron_schedule_key)}")
    print(f"  Enabled: {updated['enabled']}")
    print(f"  New cron: {updated['schedule']['cron_expression']}")
    print(f"  Description: {updated.get('description', 'N/A')}")
else:
    print(f"Error: {response.text}")

Schedule updated: asc_daily_research_digest_a99e
  Key: asc_daily_research_digest_a99e
  Enabled: True
  New cron: 0 * * * *
  Description: Generates a research digest every hour (demo)


## Step 6: Check Execution History

After a schedule runs, you can inspect its execution history. Each execution records the status,
the session it created, and any errors that occurred.

In [8]:
# Check execution history for the cron schedule
url = f"{BASE_URL}/agents/{agent_key}/schedules/{cron_schedule_key}/executions"
response = requests.get(url, headers=headers)

executions = []
if response.status_code == 200:
    data = response.json()
    executions = data.get('executions', [])
    if executions:
        print(f"Found {len(executions)} execution(s):\n")
        for execution in executions:
            print(f"  Run: {execution.get('workflow_run_id', 'N/A')}")
            print(f"    Attempt: {execution.get('attempt', 'N/A')}")
            print(f"    Status: {execution.get('status', 'N/A')}")
            print(f"    Session: {execution.get('session_key', 'N/A')}")
            print(f"    Executed at: {execution.get('executed_at', 'N/A')}")
            if execution.get('error_message'):
                print(f"    Error: {execution['error_message']}")
            print()
    else:
        print("No executions yet.")
        print("(Schedules run on their configured timing. For a newly created schedule,")
        print(" wait for the next scheduled time or check back later.)")
else:
    print(f"Error: {response.text}")

No executions yet.
(Schedules run on their configured timing. For a newly created schedule,
 wait for the next scheduled time or check back later.)


## Step 7: Inspect a Scheduled Session

When a schedule executes successfully, it creates a session. You can inspect the session's events
to see the agent's input, tool usage, and final output.

In [9]:
# If there are executions, inspect the most recent session
if executions:
    latest = executions[0]
    session_key = latest.get('session_key')
    if session_key:
        # Get the session's events to see the agent's response
        url = f"{BASE_URL}/agents/{agent_key}/sessions/{session_key}/events"
        response = requests.get(url, headers=headers)

        if response.status_code == 200:
            events = response.json().get('events', [])
            print(f"Session {session_key} events:\n")
            for event in events:
                event_type = event.get('type', 'unknown')
                if event_type == 'input_message':
                    content = event.get('content', '')
                    print(f"  Input: {content[:200]}...")
                elif event_type == 'agent_output':
                    content = event.get('content', '')
                    print(f"  Agent output:\n{content[:500]}...")
                elif event_type in ('tool_input', 'tool_output'):
                    tool = event.get('tool_configuration_name', '?')
                    print(f"  {event_type}: {tool}")
        else:
            print(f"Error: {response.text}")
    else:
        print("Latest execution has no session (may have failed before session creation)")
else:
    print("No executions to inspect yet.")
    print("The schedule will create sessions automatically when it runs.")

No executions to inspect yet.
The schedule will create sessions automatically when it runs.


## Cleanup

Delete the schedules and agent created in this notebook.

In [10]:
# Delete schedules
for schedule_key, name in [(cron_schedule_key, "Daily Research Digest"),
                           (interval_schedule_key, "Periodic Research Check")]:
    if schedule_key:
        url = f"{BASE_URL}/agents/{agent_key}/schedules/{schedule_key}"
        response = requests.delete(url, headers=headers)
        if response.status_code == 204:
            print(f"Deleted schedule: {name}")
        else:
            print(f"Error deleting {name}: {response.text}")

# Delete agent
if agent_key:
    response = requests.delete(f"{BASE_URL}/agents/{agent_key}", headers=headers)
    if response.status_code == 204:
        print(f"Deleted agent: Research Digest Generator")
    else:
        print(f"Error: {response.text}")

Deleted schedule: Daily Research Digest


Deleted schedule: Periodic Research Check


Deleted agent: Research Digest Generator
